# Laboratório — Métricas de regressão

Este laboratório reproduz os cálculos da Aula 13 e testa propriedades que costumam causar erros em avaliações reais.

**Objetivos:** conferir implementações manuais, visualizar sensibilidade a extremos, distinguir funcionais-alvo e auditar agregações.

Dados sintéticos; nenhuma rede, credencial ou arquivo externo é usado. Seed fixa: `20260908`.

## Ambiente

Dependências mínimas: Python 3.10, NumPy 1.24, pandas 2.0, Matplotlib 3.7 e scikit-learn 1.3.

No Colab, se necessário: `pip install "numpy>=1.24" "pandas>=2.0" "matplotlib>=3.7" "scikit-learn>=1.3"`.

In [ ]:
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    mean_absolute_error,
    mean_pinball_loss,
    mean_squared_error,
    r2_score,
)

SEED = 20260908
rng = np.random.default_rng(SEED)

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

## 1. Implementação manual

Usamos a convenção de resíduo `real - previsão`. MAE, MSE e RMSE são perdas; quanto menor, melhor. R² é um escore; quanto maior, melhor.

In [ ]:
def metricas_manuais(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.shape != y_pred.shape or y_true.size < 2:
        raise ValueError("Vetores devem ter o mesmo shape e pelo menos duas observações.")
    residuo = y_true - y_pred
    mae = np.mean(np.abs(residuo))
    mse = np.mean(residuo ** 2)
    rmse = np.sqrt(mse)
    sst = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = np.nan if sst == 0 else 1 - np.sum(residuo ** 2) / sst
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}


y = np.array([10.0, 12.0, 18.0])
pred = np.array([9.0, 15.0, 17.0])
manual = metricas_manuais(y, pred)
manual

In [ ]:
biblioteca = {
    "MAE": mean_absolute_error(y, pred),
    "MSE": mean_squared_error(y, pred),
    "RMSE": np.sqrt(mean_squared_error(y, pred)),
    "R2": r2_score(y, pred),
}

for nome in manual:
    np.testing.assert_allclose(manual[nome], biblioteca[nome], rtol=0, atol=1e-12)

print(pd.DataFrame({"manual": manual, "scikit-learn": biblioteca}).round(6))

**Resultado esperado:** MAE `1.666667`, MSE `3.666667`, RMSE `1.914854` e R² `0.682692`. A igualdade até `1e-12` protege contra erro de fórmula.

## 2. Mesma MAE, risco de cauda diferente

Os dois modelos abaixo erram 8 unidades no total. O modelo B concentra o erro em uma observação.

In [ ]:
erros_a = np.array([2.0, 2.0, 2.0, 2.0])
erros_b = np.array([0.0, 0.0, 0.0, 8.0])

comparacao = pd.DataFrame(
    {
        "MAE": [np.mean(np.abs(erros_a)), np.mean(np.abs(erros_b))],
        "RMSE": [np.sqrt(np.mean(erros_a ** 2)), np.sqrt(np.mean(erros_b ** 2))],
        "erro_maximo": [np.max(np.abs(erros_a)), np.max(np.abs(erros_b))],
    },
    index=["Modelo A", "Modelo B"],
)
assert comparacao.loc["Modelo A", "MAE"] == comparacao.loc["Modelo B", "MAE"] == 2
assert comparacao.loc["Modelo B", "RMSE"] == 2 * comparacao.loc["Modelo A", "RMSE"]
comparacao

## 3. Sensibilidade a um outlier

Mantemos 99 resíduos fixos e aumentamos apenas o centésimo. A curva mede efeito do tamanho, não da frequência, da contaminação.

In [ ]:
base = rng.normal(loc=0.0, scale=1.0, size=99)
magnitudes = np.linspace(0, 30, 31)
linhas = []
for magnitude in magnitudes:
    residuos = np.r_[base, magnitude]
    linhas.append(
        {
            "magnitude": magnitude,
            "MAE": np.mean(np.abs(residuos)),
            "RMSE": np.sqrt(np.mean(residuos ** 2)),
        }
    )
sensibilidade = pd.DataFrame(linhas)
sensibilidade.iloc[[0, 10, 20, 30]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sensibilidade["magnitude"], sensibilidade["MAE"], marker="o", ms=3, label="MAE")
ax.plot(sensibilidade["magnitude"], sensibilidade["RMSE"], marker="s", ms=3, label="RMSE")
ax.set(xlabel="Magnitude do único outlier", ylabel="Erro", title="RMSE reage mais a um erro extremo")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

O gráfico não prova que MAE é sempre melhor. Ele torna visível a preferência: RMSE dá influência quadrática a desvios grandes.

## 4. R² negativo e referência operacional

R² usa a média do conjunto avaliado no denominador. Uma referência congelada de produção gera outro skill score e deve receber outro nome.

In [ ]:
y_eval = np.array([1.0, 2.0, 3.0, 4.0])
pred_ruim = np.array([8.0, 8.0, 8.0, 8.0])
r2_negativo = r2_score(y_eval, pred_ruim)

pred_baseline = np.full_like(y_eval, 2.0)  # regra congelada, por exemplo média do treino
sse_modelo = np.sum((y_eval - pred_ruim) ** 2)
sse_baseline = np.sum((y_eval - pred_baseline) ** 2)
skill = 1 - sse_modelo / sse_baseline

assert r2_negativo < 0
assert not np.isclose(r2_negativo, skill)
print(f"R² padrão: {r2_negativo:.6f}")
print(f"Skill contra baseline congelado: {skill:.6f}")

## 5. MAPE perto de zero

Para evitar avisos de divisão por zero, calculamos o termo manualmente apenas para valores não nulos. A observação quase zero já basta para mostrar a instabilidade.

In [ ]:
y_quase_zero = np.array([0.001, 100.0])
pred_quase_zero = np.array([1.001, 90.0])
ape = np.abs((y_quase_zero - pred_quase_zero) / y_quase_zero)
mape = 100 * np.mean(ape)
mae = mean_absolute_error(y_quase_zero, pred_quase_zero)
wape = 100 * np.sum(np.abs(y_quase_zero - pred_quase_zero)) / np.sum(np.abs(y_quase_zero))

print(f"MAE: {mae:.6f}")
print(f"MAPE: {mape:.3f}%")
print(f"WAPE: {wape:.3f}%")
print(f"Contribuições percentuais: {(100 * ape).round(3)}")
assert mape > 50_000
assert mae < 6

Um erro absoluto de apenas 1 junto de um alvo `0.001` contribui com `100000%`. WAPE evita a divisão individual, mas responde a uma pergunta agregada e pode esconder itens pequenos.

## 6. Cada perda seleciona um funcional

Geramos uma distribuição assimétrica. Em uma grade de previsões constantes, MSE deve atingir o mínimo perto da média; MAE, perto da mediana; pinball de 0,9, perto do quantil 90%.

In [ ]:
amostra = rng.lognormal(mean=1.0, sigma=0.8, size=20_000)
grade = np.linspace(np.quantile(amostra, 0.01), np.quantile(amostra, 0.99), 400)

maes = np.array([mean_absolute_error(amostra, np.full_like(amostra, a)) for a in grade])
mses = np.array([mean_squared_error(amostra, np.full_like(amostra, a)) for a in grade])
pinballs = np.array([mean_pinball_loss(amostra, np.full_like(amostra, a), alpha=0.9) for a in grade])

estimativas = {
    "argmin_MAE": grade[np.argmin(maes)],
    "mediana": np.median(amostra),
    "argmin_MSE": grade[np.argmin(mses)],
    "media": np.mean(amostra),
    "argmin_pinball_0.9": grade[np.argmin(pinballs)],
    "quantil_0.9": np.quantile(amostra, 0.9),
}
pd.Series(estimativas).round(4)

In [ ]:
passo = grade[1] - grade[0]
assert abs(estimativas["argmin_MAE"] - estimativas["mediana"]) <= 2 * passo
assert abs(estimativas["argmin_MSE"] - estimativas["media"]) <= 2 * passo
assert abs(estimativas["argmin_pinball_0.9"] - estimativas["quantil_0.9"]) <= 2 * passo
print("Funcionais recuperados dentro da resolução da grade.")

## 7. Agregação correta do RMSE

O lote grande tem 100 erros de magnitude 1; o pequeno, um erro de magnitude 10.

In [ ]:
lote_grande = np.ones(100)
lote_pequeno = np.array([10.0])

rmse_grande = np.sqrt(np.mean(lote_grande ** 2))
rmse_pequeno = np.sqrt(np.mean(lote_pequeno ** 2))
media_ingenua = np.mean([rmse_grande, rmse_pequeno])
rmse_global = np.sqrt(np.mean(np.r_[lote_grande, lote_pequeno] ** 2))

np.testing.assert_allclose(rmse_global, np.sqrt(200 / 101), atol=1e-12)
print(f"Média ingênua dos RMSEs: {media_ingenua:.6f}")
print(f"RMSE global correto: {rmse_global:.6f}")

## 8. Micro versus macro por grupo

O grupo grande possui escala maior e mais observações. Comparamos erro absoluto global e erro relativo médio por grupo.

In [ ]:
grupos = pd.DataFrame(
    {
        "grupo": ["alto_volume"] * 100 + ["baixo_volume"] * 10,
        "real": np.r_[np.full(100, 100.0), np.full(10, 5.0)],
        "prev": np.r_[np.full(100, 90.0), np.full(10, 2.0)],
    }
)
grupos["erro_abs"] = np.abs(grupos["real"] - grupos["prev"])
micro_mae = grupos["erro_abs"].mean()
por_grupo = grupos.groupby("grupo", sort=True).agg(
    n=("real", "size"),
    mae=("erro_abs", "mean"),
    media_real=("real", "mean"),
)
por_grupo["erro_relativo"] = por_grupo["mae"] / por_grupo["media_real"]
macro_relativo = por_grupo["erro_relativo"].mean()

print(f"MAE micro: {micro_mae:.6f}")
print(f"Erro relativo macro: {macro_relativo:.6f}")
por_grupo

Não existe agregação universalmente correta. O relatório deve dizer se a unidade de decisão é uma observação, um produto, uma loja ou outro grupo.

## 9. Verificações finais

In [ ]:
assert SEED == 20260908
assert len(y) == len(pred)
assert manual["RMSE"] >= manual["MAE"]
assert comparacao.loc["Modelo B", "erro_maximo"] == 8
assert sensibilidade.iloc[-1]["RMSE"] > sensibilidade.iloc[-1]["MAE"]
assert r2_negativo < 0
assert np.isfinite(mape)
assert rmse_global < media_ingenua
assert set(por_grupo.index) == {"alto_volume", "baixo_volume"}
assert not any(issubclass(w.category, Warning) for w in [])
print("Todas as verificações passaram.")

## Conclusões verificadas

- Implementação manual e biblioteca coincidem até `1e-12`.
- Mesma MAE não implica o mesmo risco de cauda: os RMSEs foram `2` e `4`.
- Um único outlier afeta RMSE mais rapidamente que MAE.
- R² pode ser negativo e não é igual a skill contra baseline congelado.
- MAPE pode explodir com alvo quase zero.
- MAE, MSE e pinball recuperam mediana, média e quantil, respectivamente.
- A média de RMSEs por lote (`5.5`) difere do RMSE global (`sqrt(200/101)`).
- Micro e macroagregação representam unidades de decisão distintas.

Antes de comparar modelos reais, registre métrica primária, baseline, grupos, pesos e regra de agregação.